In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import numpy as np
import tqdm
import matplotlib.pyplot as plt
from string import ascii_uppercase, ascii_lowercase

from glob import glob
import os
import pickle

from scipy.special import softmax
import pandas as pd
import bokeh.plotting
from bokeh.transform import linear_cmap
from bokeh.plotting import figure, show
from bokeh.palettes import viridis
from transformers import AutoTokenizer, AutoModelForMaskedLM
from matplotlib.colors import to_hex
import tqdm.notebook
bokeh.io.output_notebook()

MODEL_NAME = "tattabio/gLM2_650M"
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
MODEL = AutoModelForMaskedLM.from_pretrained(MODEL_NAME, trust_remote_code=True).eval().to(DEVICE)
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# set desired alphabet order
ALPHABET = "AFILVMWYDEKRHNQSTGPC"
MASK_ID = TOKENIZER.convert_tokens_to_ids('<mask>')
ESM_ALPHABET = TOKENIZER.convert_ids_to_tokens(range(4, 25))
ALPHABET_MAP = [ESM_ALPHABET.index(a) for a in ALPHABET]

cmap = plt.colormaps["bwr_r"]
bwr_r = [to_hex(cmap(i)) for i in np.linspace(0, 1, 256)]
cmap = plt.colormaps["gray_r"]
gray = [to_hex(cmap(i)) for i in np.linspace(0, 1, 256)]

ASCII_LIST = list(ascii_uppercase+ascii_lowercase)
TQDM_BAR_FORMAT = '{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]'

def get_categorical_jacobian(seqs, prepend_seq="<+>", fast=False):
  # ∂in/∂out

  xs = []
  masks = []
  for seq in seqs:
    if len(seq) > 0:
      x = TOKENIZER([prepend_seq+seq])["input_ids"][0]
      mask = np.pad(np.full(len(seq),True),[len(x)-len(seq),0])
      xs.append(x)
      masks.append(mask)

  x = torch.tensor(np.concatenate(xs,-1)[None]).to(DEVICE)
  mask = np.concatenate(masks,-1)

  with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
    f = lambda x: MODEL(x).logits[:, :, 4:24].detach().cpu().numpy()
    fx = f(x.to(DEVICE))[0][mask]

    ln = sum(mask)
    if fast:
      fx_h = np.zeros([ln, 1 , ln, 20], dtype=np.float32)
      x = x.to(DEVICE)
    else:
      fx_h = np.zeros([ln, 20, ln, 20], dtype=np.float32)
      x = torch.tile(x, [20, 1]).to(DEVICE)

    with tqdm.notebook.tqdm(total=ln, bar_format=TQDM_BAR_FORMAT) as pbar:
      i = 0
      for n in range(len(mask)):  # for each position
        x_h = torch.clone(x)
        if mask[n]:
          if fast:
            x_h[:, n] = MASK_ID
          else:
            # mutate to all 20 aa
            x_h[:, n] = torch.arange(4, 24)
          fx_h[i] = f(x_h)[:,mask]
          i += 1
          pbar.update(1)

    return fx_h - fx

def jac_to_con(jac, symm=True, center=True, diag="remove", apc=True):

  X = jac.copy()
  Lx,Ax,Ly,Ay = X.shape
  if Ax == 20:
    X = X[:,ALPHABET_MAP,:,:]

  if Ay == 20:
    X = X[:,:,:,ALPHABET_MAP]
    if symm and Ax == 20:
      X = (X + X.transpose(2,3,0,1))/2

  if center:
    for i in range(4):
      if X.shape[i] > 1:
        X -= X.mean(i,keepdims=True)

  contacts = np.sqrt(np.square(X).sum((1,3)))

  if symm and (Ax != 20 or Ay != 20):
    contacts = (contacts + contacts.T)/2

  if diag == "remove":
    np.fill_diagonal(contacts,0)

  if diag == "normalize":
    contacts_diag = np.diag(contacts)
    contacts = contacts / np.sqrt(contacts_diag[:,None] * contacts_diag[None,:])

  if apc:
    ap = contacts.sum(0,keepdims=True) * contacts.sum(1, keepdims=True) / contacts.sum()
    contacts = contacts - ap

  if diag == "remove":
    np.fill_diagonal(contacts,0)

  return {"jac":X, "contacts":contacts}

def plot_ticks(Ls, axes=None, chain_list=None):
  if chain_list is None:
    chain_list = ASCII_LIST
  if axes is None: axes = plt.gca()
  Ln = sum(Ls)
  L_prev = 0
  for L_i in Ls[:-1]:
    L = L_prev + L_i
    L_prev += L_i
    plt.plot([0,Ln],[L,L],color="black")
    plt.plot([L,L],[0,Ln],color="black")
  ticks = np.cumsum([0]+Ls)
  ticks = (ticks[1:] + ticks[:-1])/2
  axes.set_yticks(ticks)
  axes.set_yticklabels(chain_list[:len(ticks)])

def contact_to_dataframe(con, idx=None):
  sequence_length = con.shape[0]
  if idx is None:
    idx = [str(i) for i in np.arange(1, sequence_length + 1)]
  df = pd.DataFrame(con, index=idx, columns=idx)
  df = df.stack().reset_index()
  df.columns = ['i', 'j', 'value']
  return df

def pair_to_dataframe(pair):
  df = pd.DataFrame(pair, index=list(ALPHABET), columns=list(ALPHABET))
  df = df.stack().reset_index()
  df.columns = ['aa_i', 'aa_j', 'value']
  return df

# All gremlin examples

In [ ]:
# Load target data
target_data_file = "../../data/Figure2_heterooligomer_contact_prediction/target_data.pkl"
with open(target_data_file, "rb") as oFile:
    target_data_d = pickle.load(oFile)

In [16]:
target_data_d['4HR7_A_4HR7_B']['chain_break']

449

In [ ]:
for target in tqdm.tqdm(target_data_d):
    msa_file = target_data_d[target]['msa_file']
    with open(msa_file, "r") as oFile:
        query_seq = oFile.readlines()[1].strip()
    seperator = "<+>"
    fast = False
    chain_break_idx = target_data_d[target]['chain_break']
    seq_A = query_seq[:chain_break_idx]
    seq_B = query_seq[chain_break_idx:]

    seqs = []
    Ls = []
    chain_idx = []
    residue_idx = []
    for n,seq in enumerate([seq_A,seq_B]):
        seq = seq.replace(" ","").upper()
        seq = ''.join([i for i in seq if i.isalpha()])
        if len(seq) > 0:
            seqs.append(seq)
            Ls.append(len(seq))
            chain_idx.append(ASCII_LIST[n])
            residue_idx += [f"{ASCII_LIST[n]}{i+1}" for i in range(len(seq))]

    sequence = "".join(seqs)

    jac = get_categorical_jacobian(
        seqs,
        prepend_seq=seperator,
        fast=fast
    )
    con = jac_to_con(jac)
    L = con["contacts"].shape[0]

    ############ SAVE #####################
    np.savetxt(f"results/glm2/pred_contacts_full/{target}_coevolution.txt",con["contacts"])

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/217 [elapsed: 00:00 remaining: ?]

 47%|████▋     | 15/32 [00:14<00:16,  1.04it/s]

  0%|          | 0/740 [elapsed: 00:00 remaining: ?]

 50%|█████     | 16/32 [03:34<04:52, 18.25s/it]

...

 94%|█████████▍| 30/32 [27:34<03:23, 101.67s/it]

  0%|          | 0/564 [elapsed: 00:00 remaining: ?]

 97%|█████████▋| 31/32 [29:25<01:44, 104.44s/it]

  0%|          | 0/927 [elapsed: 00:00 remaining: ?]

100%|██████████| 32/32 [34:51<00:00, 65.34s/it] 
